##### Customer Sample:

In [0]:
customer_df = spark.read.format("csv")\
              .option("header",True)\
              .option("inferSchema",True)\
              .load("/Volumes/pysparkdbt/source/source_data/customers/")
display(customer_df)

In [0]:
customer_schema = customer_df.schema
customer_schema

#### Spark Streaming:
##### Dynamic Method for all the entities:

In [0]:
entities = ['customers','trips','drivers','vehicles','payments','locations']

In [0]:
for entity in entities:

        batch_df = spark.read.format("csv")\
              .option("header",True)\
              .option("inferSchema",True)\
              .load(f"/Volumes/pysparkdbt/source/source_data/{entity}/")

        schema_entity = batch_df.schema

        customer_stream_df = spark.readStream.format("csv")\
                            .option("header",True)\
                            .schema(schema_entity)\
                            .load(f"/Volumes/pysparkdbt/source/source_data/{entity}/")

        customer_stream_df.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation",f"/Volumes/pysparkdbt/bronze/checkpoint/{entity}")\
            .trigger(once=True)\
            .toTable(f"pysparkdbt.bronze.{entity}")